# Session 11

## Decision Trees

In [2]:
"""
Decision tree demo for regression and classification, written from the
algorithms of the lecture notes (Algorithm 1, the sorted sweep with
incremental statistics, and Algorithm 2, recursive growing with stopping
rules), plus weakest-link cost-complexity pruning.

The demo shows
  1. the splits one after the other, in order of decreasing impurity decrease,
     with the training loss after every split (Proposition 16.1: it never rises),
  2. the held-out loss after every split, which rises again once the tree overfits,
  3. the regularisation options max_depth, min_samples_leaf,
     min_impurity_decrease and ccp_alpha,
  4. bias^2 and variance of the tree as functions of the regularisation,
     estimated by redrawing the training set many times.

Impurity decrease and ccp_alpha use the units of scikit-learn, that is the
notes' quantities divided by the number n of training points. The script checks
its trees against scikit-learn before it plots anything.
"""
import heapq
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

SEED = 0
SIGMA = 0.5
f_true = lambda x: np.sin(2 * x)


class Node:
    def __init__(self, idx, depth, nid):
        self.idx, self.depth, self.id = idx, depth, nid
        self.feat = self.thr = self.left = self.right = None
        self.cost = self.gain = self.value = None
        self.collapsed = False


class Tree:
    """criterion 'var' (regression) or 'gini' (classification).
    cost(R) = n(R) * Q(R); the gain of a split is n(R) * Delta."""

    def __init__(self, criterion="var", max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, min_impurity_decrease=0.0, ccp_alpha=0.0):
        self.criterion, self.max_depth = criterion, max_depth
        self.min_samples_split, self.min_samples_leaf = min_samples_split, min_samples_leaf
        self.min_impurity_decrease, self.ccp_alpha = min_impurity_decrease, ccp_alpha

    # ----- fitting -------------------------------------------------------
    def fit(self, X, y):
        self.X, self.y, self.n = X, y, len(y)
        if self.criterion == "gini":
            self.classes = np.unique(y)
            self.Y = (y[:, None] == self.classes[None, :]).astype(float)
        self.nodes = []
        self.root = self._grow(np.arange(self.n), 0)
        self._collapse_path()
        self.apply_alpha(self.ccp_alpha)
        return self

    def _cost(self, idx):
        n = len(idx)
        if self.criterion == "var":
            ys = self.y[idx]
            return float((ys ** 2).sum() - ys.sum() ** 2 / n)
        c = self.Y[idx].sum(0)
        return float(n - (c ** 2).sum() / n)

    def _value(self, idx):
        if self.criterion == "var":
            return float(self.y[idx].mean())
        return self.classes[np.argmax(self.Y[idx].sum(0))]

    def _best_split(self, idx, cost):
        n, best = len(idx), (-np.inf, None, None)
        r = np.arange(1, n)
        for j in range(self.X.shape[1]):
            order = idx[np.argsort(self.X[idx, j], kind="stable")]
            xs = self.X[order, j]
            ok = (xs[:-1] < xs[1:]) & (r >= self.min_samples_leaf) & (n - r >= self.min_samples_leaf)
            if not ok.any():
                continue
            if self.criterion == "var":
                ys = self.y[order]
                cs, cs2 = np.cumsum(ys)[:-1], np.cumsum(ys ** 2)[:-1]
                S, S2 = ys.sum(), (ys ** 2).sum()
                cost_l = cs2 - cs ** 2 / r
                cost_r = (S2 - cs2) - (S - cs) ** 2 / (n - r)
            else:
                cn = np.cumsum(self.Y[order], axis=0)[:-1]
                tot = self.Y[order].sum(0)
                cost_l = r - (cn ** 2).sum(1) / r
                cost_r = (n - r) - ((tot - cn) ** 2).sum(1) / (n - r)
            gain = np.where(ok, cost - (cost_l + cost_r), -np.inf)
            k = int(np.argmax(gain))
            if gain[k] > best[0]:
                best = (float(gain[k]), j, 0.5 * (xs[k] + xs[k + 1]))
        return best

    def _grow(self, idx, depth):
        node = Node(idx, depth, len(self.nodes))
        self.nodes.append(node)
        node.cost, node.value = self._cost(idx), self._value(idx)
        if (node.cost <= 1e-12 or len(idx) < self.min_samples_split
                or (self.max_depth is not None and depth >= self.max_depth)):
            return node
        gain, j, thr = self._best_split(idx, node.cost)
        if j is None or gain / self.n < self.min_impurity_decrease:
            return node
        node.feat, node.thr, node.gain = j, thr, gain
        mask = self.X[idx, j] < thr
        node.left = self._grow(idx[mask], depth + 1)
        node.right = self._grow(idx[~mask], depth + 1)
        return node

    # ----- weakest-link pruning (Proposition 16.8) -----------------------
    def _subtree(self, node):
        if node.left is None or node.collapsed:
            return node.cost, 1
        cl, nl = self._subtree(node.left)
        cr, nr = self._subtree(node.right)
        return cl + cr, nl + nr

    def _collapse_path(self):
        """Collapse the node with the smallest g(t) again and again.
        path[k] = (effective alpha, node), alpha in units of the notes / n."""
        for nd in self.nodes:
            nd.collapsed = False
        self.path, prev = [], 0.0
        while True:
            best = None
            for nd in self.nodes:
                if nd.left is None or nd.collapsed:
                    continue
                c, nleaves = self._subtree(nd)
                g = (nd.cost - c) / (nleaves - 1) / self.n
                if best is None or g < best[0]:
                    best = (g, nd)
            if best is None:
                break
            prev = max(prev, best[0])
            best[1].collapsed = True
            self.path.append((prev, best[1]))
        for nd in self.nodes:
            nd.collapsed = False

    def apply_alpha(self, alpha):
        for nd in self.nodes:
            nd.collapsed = False
        for a, nd in self.path:
            if a <= alpha:
                nd.collapsed = True
            else:
                break
        return self

    # ----- prediction and bookkeeping ------------------------------------
    def predict(self, X, applied=None):
        out = np.empty(len(X), dtype=float if self.criterion == "var" else self.classes.dtype)
        self._predict(self.root, np.arange(len(X)), X, out, applied)
        return out

    def _is_leaf(self, nd, applied):
        return nd.left is None or nd.collapsed or (applied is not None and nd.id not in applied)

    def _predict(self, nd, rows, X, out, applied):
        if len(rows) == 0:
            return
        if self._is_leaf(nd, applied):
            out[rows] = nd.value
            return
        m = X[rows, nd.feat] < nd.thr
        self._predict(nd.left, rows[m], X, out, applied)
        self._predict(nd.right, rows[~m], X, out, applied)

    def n_leaves(self):
        return self._subtree(self.root)[1]

    def split_order(self):
        """Best-first replay of the splits of the grown tree. A node can only be
        split after its parent, and among the available splits the one with the
        largest decrease comes first."""
        order, heap = [], []
        def push(nd):
            if nd.left is not None and not nd.collapsed:
                heapq.heappush(heap, (-nd.gain, nd.id))
        push(self.root)
        while heap:
            _, nid = heapq.heappop(heap)
            nd = self.nodes[nid]
            order.append(nd)
            push(nd.left); push(nd.right)
        return order


# ---------------------------------------------------------------------------
def make_regression(n, rng):
    x = rng.uniform(0, 2 * np.pi, size=(n, 1))
    return x, f_true(x[:, 0]) + rng.normal(0, SIGMA, size=n)


def check_against_sklearn():
    rng = np.random.default_rng(1)
    X, y = make_regression(80, rng)
    grid = np.linspace(0, 2 * np.pi, 400).reshape(-1, 1)
    settings = [dict(), dict(max_depth=3), dict(min_samples_leaf=8), dict(min_samples_split=20),
                dict(min_impurity_decrease=0.005), dict(ccp_alpha=0.01)]
    worst = 0.0
    for kw in settings:
        ours = Tree("var", **kw).fit(X, y).predict(grid)
        ref = DecisionTreeRegressor(random_state=0, **kw).fit(X, y).predict(grid)
        worst = max(worst, np.abs(ours - ref).max())
    Xc, yc = make_moons(200, noise=0.3, random_state=2)
    gx = np.random.default_rng(3).uniform(-2, 3, size=(500, 2))
    wrong = 0
    for kw in [dict(), dict(max_depth=3), dict(min_samples_leaf=5)]:
        wrong += int((Tree("gini", **kw).fit(Xc, yc).predict(gx)
                      != DecisionTreeClassifier(random_state=0, **kw).fit(Xc, yc).predict(gx)).sum())
    print(f"check against scikit-learn: regression max |difference| = {worst:.2e}, "
          f"classification disagreements = {wrong} of 1500")


def step_table(tree, order, Xte, yte, n_rows=12):
    print("\nStep by step, regression, no regularisation (best-first order of the splits)")
    print(" step  depth  n(R)  threshold   decrease n(R)*Delta   train MSE   test MSE")
    cost, applied = tree.root.cost, set()
    print(f"    0      0  {tree.n:4d}      -                 -            {cost / tree.n:7.4f}   "
          f"{np.mean((yte - tree.predict(Xte, applied)) ** 2):7.4f}")
    for k, nd in enumerate(order[:n_rows], 1):
        cost -= nd.gain
        applied.add(nd.id)
        print(f"{k:5d}  {nd.depth:5d}  {len(nd.idx):4d}  {nd.thr:9.3f}   {nd.gain:18.4f}   "
              f"{cost / tree.n:9.4f}   {np.mean((yte - tree.predict(Xte, applied)) ** 2):8.4f}")


def curves(tree, order, Xte, yte, classification=False):
    applied, cost = set(), tree.root.cost
    rows = []
    for k in range(len(order) + 1):
        if k > 0:
            cost -= order[k - 1].gain
            applied.add(order[k - 1].id)
        pred_tr, pred_te = tree.predict(tree.X, applied), tree.predict(Xte, applied)
        if classification:
            rows.append((cost / tree.n, np.mean(pred_tr != tree.y), np.mean(pred_te != yte)))
        else:
            rows.append((cost / tree.n, np.mean((tree.y - pred_tr) ** 2), np.mean((yte - pred_te) ** 2)))
    return np.array(rows), np.array([nd.gain / tree.n for nd in order])


def fig_steps_regression(tree, order, Xte, yte, fname):
    grid = np.linspace(0, 2 * np.pi, 400).reshape(-1, 1)
    ks = [0, 1, 2, 3, 5, 10, 25, len(order)]
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
    for ax, k in zip(axes.ravel(), ks):
        applied = {nd.id for nd in order[:k]}
        ax.plot(grid, f_true(grid[:, 0]), "g:", label="true $f$")
        ax.plot(tree.X[:, 0], tree.y, ".", color="grey", ms=6)
        ax.plot(grid, tree.predict(grid, applied), "r-", lw=1.8)
        tr = np.mean((tree.y - tree.predict(tree.X, applied)) ** 2)
        te = np.mean((yte - tree.predict(Xte, applied)) ** 2)
        ax.set_title(f"{k} " + ("split" if k == 1 else "splits") + f", {k + 1} " + ("leaf" if k == 0 else "leaves") + f"\ntrain MSE {tr:.3f}, test MSE {te:.3f}", fontsize=10)
        ax.set_xlabel("$x$")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle("Regression tree, grown split by split in order of decreasing impurity decrease")
    fig.tight_layout(); fig.savefig(fname, dpi=130); plt.close(fig)


def fig_steps_classification(tree, order, Xte, yte, fname):
    gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 250), np.linspace(-1.2, 1.8, 250))
    grid = np.c_[gx.ravel(), gy.ravel()]
    ks = [1, 2, 4, 8, len(order)]
    fig, axes = plt.subplots(1, len(ks), figsize=(18, 3.9))
    for ax, k in zip(axes, ks):
        applied = {nd.id for nd in order[:k]}
        Z = tree.predict(grid, applied).reshape(gx.shape)
        ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5], colors=["#abc9ea", "#efb3b3"], alpha=0.6)
        ax.scatter(tree.X[:, 0], tree.X[:, 1], c=np.where(tree.y == 1, "tab:red", "tab:blue"), s=12)
        tr = np.mean(tree.predict(tree.X, applied) != tree.y)
        te = np.mean(tree.predict(Xte, applied) != yte)
        ax.set_title(f"{k} " + ("split" if k == 1 else "splits") + f", {k + 1} leaves\ntrain error {tr:.2f}, test error {te:.2f}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Classification tree (Gini impurity), grown split by split")
    fig.tight_layout(); fig.savefig(fname, dpi=130); plt.close(fig)


def fig_loss(reg, cls, fname):
    (rc, rg), (cc, cg) = reg, cls
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    k = np.arange(len(rc))
    ax = axes[0, 0]
    ax.plot(k, rc[:, 1], label="training MSE"); ax.plot(k, rc[:, 2], label="test MSE")
    ax.axhline(SIGMA ** 2, color="grey", ls=":", label="noise $\\sigma^2$")
    ax.axvline(np.argmin(rc[:, 2]), color="k", ls="--", lw=0.8)
    ax.set_title("regression, loss after each split"); ax.set_xlabel("number of splits"); ax.legend()
    axes[0, 1].semilogy(np.arange(1, len(rg) + 1), rg, ".-")
    axes[0, 1].set_title("regression, decrease $n(R)\\Delta/n$ of each split"); axes[0, 1].set_xlabel("split")
    k = np.arange(len(cc)); ax = axes[1, 0]
    ax.plot(k, cc[:, 0], label="Gini impurity of the tree"); ax.plot(k, cc[:, 1], label="training error")
    ax.plot(k, cc[:, 2], label="test error"); ax.axvline(np.argmin(cc[:, 2]), color="k", ls="--", lw=0.8)
    ax.set_title("classification, loss after each split"); ax.set_xlabel("number of splits"); ax.legend()
    axes[1, 1].semilogy(np.arange(1, len(cg) + 1), np.maximum(cg, 1e-6), ".-")
    axes[1, 1].set_title("classification, decrease of each split"); axes[1, 1].set_xlabel("split")
    fig.suptitle("The training loss never rises (Proposition 16.1). The test loss has a minimum, shown dashed.")
    fig.tight_layout(); fig.savefig(fname, dpi=130); plt.close(fig)


def fig_regularisation(X, y, Xte, yte, fname):
    settings = [("none", {}), ("max_depth=3", dict(max_depth=3)),
                ("min_samples_leaf=8", dict(min_samples_leaf=8)),
                ("min_impurity_decrease=0.005", dict(min_impurity_decrease=0.005)),
                ("ccp_alpha=0.01", dict(ccp_alpha=0.01))]
    grid = np.linspace(0, 2 * np.pi, 400).reshape(-1, 1)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    stats = []
    print("\nRegularisation options (regression)")
    print(" setting                        leaves   train MSE   test MSE")
    for ax, (name, kw) in zip(axes.ravel(), settings):
        t = Tree("var", **kw).fit(X, y)
        tr, te = np.mean((y - t.predict(X)) ** 2), np.mean((yte - t.predict(Xte)) ** 2)
        stats.append((name, tr, te))
        print(f" {name:30s} {t.n_leaves():6d}   {tr:9.4f}  {te:9.4f}")
        ax.plot(grid, f_true(grid[:, 0]), "g:"); ax.plot(X[:, 0], y, ".", color="grey")
        ax.plot(grid, t.predict(grid), "r-", lw=1.8)
        ax.set_title(f"{name}\n{t.n_leaves()} leaves, train {tr:.3f}, test {te:.3f}", fontsize=10)
    ax = axes[1, 2]
    w = np.arange(len(stats))
    ax.bar(w - 0.2, [s[1] for s in stats], 0.4, label="train MSE")
    ax.bar(w + 0.2, [s[2] for s in stats], 0.4, label="test MSE")
    ax.axhline(SIGMA ** 2, color="grey", ls=":"); ax.set_xticks(w)
    ax.set_xticklabels([s[0].split("=")[0] for s in stats], rotation=20, fontsize=8)
    ax.set_title("training and test MSE per option", fontsize=11); ax.legend()
    fig.tight_layout(); fig.savefig(fname, dpi=130); plt.close(fig)


def bias_variance(R=100, n=80):
    xg = np.linspace(0.3, 2 * np.pi - 0.3, 60).reshape(-1, 1)
    truth = f_true(xg[:, 0])
    depths = list(range(1, 11))
    leaves = [1, 2, 3, 5, 8, 12, 20, 30, 40]
    alphas = np.logspace(-4, -0.7, 10)
    P = {"depth": np.zeros((R, len(depths), len(xg))), "leaf": np.zeros((R, len(leaves), len(xg))),
         "alpha": np.zeros((R, len(alphas), len(xg)))}
    rng = np.random.default_rng(SEED)
    y_new = np.zeros((R, len(xg)))
    for r in range(R):
        X, y = make_regression(n, rng)
        y_new[r] = truth + rng.normal(0, SIGMA, size=len(xg))
        for i, d in enumerate(depths):
            P["depth"][r, i] = Tree("var", max_depth=d).fit(X, y).predict(xg)
        for i, m in enumerate(leaves):
            P["leaf"][r, i] = Tree("var", min_samples_leaf=m).fit(X, y).predict(xg)
        full = Tree("var").fit(X, y)
        for i, a in enumerate(alphas):
            P["alpha"][r, i] = full.apply_alpha(a).predict(xg)
    out = {}
    for key, vals in [("depth", depths), ("leaf", leaves), ("alpha", alphas)]:
        pr = P[key]
        var = pr.var(axis=0, ddof=1)                           # (setting, x)
        bias2 = ((pr.mean(0) - truth) ** 2) - var / R          # removes the sampling noise of the mean
        mse = ((y_new[:, None, :] - pr) ** 2).mean(axis=(0, 2))
        out[key] = (np.array(vals), bias2.mean(1), var.mean(1), mse)
    return out


def fig_bias_variance(out, fname):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    titles = {"depth": "max_depth", "leaf": "min_samples_leaf", "alpha": "ccp_alpha"}
    print("\nBias and variance (regression, 100 training sets of 80 points)")
    for ax, key in zip(axes, ["depth", "leaf", "alpha"]):
        v, b2, var, mse = out[key]
        ax.plot(v, b2, "o-", label="bias$^2$"); ax.plot(v, var, "s-", label="variance")
        ax.plot(v, SIGMA ** 2 + b2 + var, "k-", label="$\\sigma^2$ + bias$^2$ + variance")
        ax.plot(v, mse, "k.", ms=9, label="test MSE, simulated")
        ax.set_xlabel(titles[key]); ax.set_title(f"regularisation by {titles[key]}")
        if key != "depth":
            ax.set_xscale("log")
        ax.legend(fontsize=8)
        print(f" {titles[key]}")
        print("   value      bias^2   variance   sigma^2+b^2+v   test MSE")
        for i in range(len(v)):
            print(f"   {v[i]:8.4g}  {b2[i]:7.4f}  {var[i]:8.4f}   {SIGMA**2 + b2[i] + var[i]:12.4f}   {mse[i]:8.4f}")
    fig.suptitle("Weaker regularisation lowers the bias and, beyond very small trees, raises the variance")
    fig.tight_layout(); fig.savefig(fname, dpi=130); plt.close(fig)


if __name__ == "__main__":
    check_against_sklearn()
    rng = np.random.default_rng(SEED)
    X, y = make_regression(80, rng)
    Xte, yte = make_regression(2000, np.random.default_rng(99))

    reg = Tree("var").fit(X, y)
    order = reg.split_order()
    step_table(reg, order, Xte, yte)
    fig_steps_regression(reg, order, Xte, yte, "tree_steps_regression.png")

    Xc, yc = make_moons(150, noise=0.3, random_state=4)
    Xct, yct = make_moons(2000, noise=0.3, random_state=5)
    cls = Tree("gini").fit(Xc, yc)
    corder = cls.split_order()
    fig_steps_classification(cls, corder, Xct, yct, "tree_steps_classification.png")

    fig_loss(curves(reg, order, Xte, yte), curves(cls, corder, Xct, yct, True), "tree_loss_per_split.png")
    fig_regularisation(X, y, Xte, yte, "tree_regularisation.png")
    fig_bias_variance(bias_variance(), "tree_bias_variance.png")

check against scikit-learn: regression max |difference| = 3.33e-16, classification disagreements = 0 of 1500

Step by step, regression, no regularisation (best-first order of the splits)
 step  depth  n(R)  threshold   decrease n(R)*Delta   train MSE   test MSE
    0      0    80      -                 -             0.8155    0.7479
    1      0    80      4.765              14.2572      0.6373     0.6145
    2      1    60      3.351               9.2359      0.5219     0.5495
    3      2    40      1.786              19.3550      0.2799     0.3924
    4      2    20      4.265               2.3369      0.2507     0.3855
    5      1    20      5.780               2.1879      0.2234     0.3957
    6      3    20      2.972               1.9940      0.1984     0.3588
    7      4    17      2.271               1.7373      0.1767     0.3734
    8      2    11      5.465               1.3143      0.1603     0.3763
    9      5    10      2.719               1.0499      0.1472     0.3689